# **Convolutional Neural Network**

# **Import the Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

# **Download Dataset**

In [2]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='data', download=True, train=True, transform=transform)
test_dataset = datasets.MNIST(root='data', download=True, train=False, transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 341kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.3MB/s]


# **Create Dataloader**

In [3]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# **Model Building**

In [4]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

        nn.ReLU(),

        nn.MaxPool2d(2),

        nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        ),

        nn.ReLU(),

        nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(32 * 7 * 7, 128),

        nn.ReLU(),

        nn.Linear(128, 10)
    )


  def forward(self, x):

    x = self.features(x)

    x = self.classifier(x)

    return x

# **Training Loop**

In [6]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):

  model.train()

  running_loss = 0.0

  for images, labels in train_loader:

    outputs = model(images)

    loss = criterion(outputs, labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    running_loss += loss.item()

avg_loss = running_loss / len(train_loader)

print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}")

Epoch [5/5]  Loss: 0.0263


# **Model Evaluation**

In [7]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.96%
